# TextCNN B1 — Cyber-aware EDA Config_4

**Scenario:** `B1`  
**Seed:** `42`  
**Backbone:** TextCNN  
**Loss:** ASL (`gamma_neg=4.0`, `gamma_pos=1.0`, `clip=0.05`, `eps=1e-8`)  
**Mục tiêu:** Original + Cyber EDA (Config_4) → Global Threshold.

Hỗ trợ chọn dataset qua `DATASET_SUBSET = 'joint' | 'cti_to_mitre' | 'tram'`. Checkpoint theo Validation Macro-F1@0.5, threshold tune validation-only, test once.


In [ ]:
from pathlib import Path

SCENARIO = "B1"
DATASET_SUBSET = "joint"  # Options: "joint" (default), "cti_to_mitre", "tram"
DATASET_ROOT = "/kaggle/input"
RESULTS_ROOT = "/kaggle/working/results" if Path("/kaggle/working").exists() else "results"


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


In [ ]:
import os, re, ast, json, time, random, pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, f1_score, hamming_loss

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = {
    "seed": SEED,
    "max_length": 384,
    "max_vocab": 50000,
    "embedding_dim": 300,
    "num_filters": 128,
    "kernel_sizes": [3, 4, 5],
    "dropout": 0.5,
    "batch_size": 64,
    "eval_batch_size": 128,
    "stage1_epochs": 15,
    "stage2_epochs": 5,
    "stage1_lr": 1e-3,
    "stage2_lr": 1e-4,
    "weight_decay": 1e-5,
    "max_grad_norm": 5.0,
    "early_stopping_patience_stage1": 3,
    "early_stopping_patience_stage2": 2,
    "threshold_min": 0.05,
    "threshold_max": 0.95,
    "threshold_step": 0.01,
    "min_val_support_per_label": 5,
    "minority_groups": ["Medium", "Tail"],
    "num_workers": 2,
    "gamma_neg": 4.0,
    "gamma_pos": 1.0,
    "asl_clip": 0.05,
    "asl_eps": 1e-8,
}

PAD_IDX = 0
UNK_IDX = 1
TOKEN_RE = re.compile(r"[a-z0-9_./:\\-]+|[^\s]", re.I)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def detect_col(df, names, required=True):
    low = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    if required:
        raise KeyError(f"Missing columns {names}; available={list(df.columns)}")
    return None


def parse_labels(x):
    if isinstance(x, (list, tuple, set, np.ndarray)):
        return [str(v).strip() for v in x if str(v).strip()]
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    if s.startswith("[") and s.endswith("]"):
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, (list, tuple, set)):
                return [str(v).strip() for v in obj if str(v).strip()]
        except Exception:
            pass
    for sep in ("|", ";", ","):
        if sep in s:
            return [v.strip().strip("'\"") for v in s.split(sep) if v.strip()]
    return [s.strip("'\"")]


def normalize_text(x):
    return re.sub(r"\s+", " ", str(x).lower()).strip()


def find_dataset_file(root, names, subset="joint", required=True):
    root = Path(root)
    search_dirs = [
        Path("dataset/processed") / subset,
        Path("dataset/processed"),
        root / subset,
        root / f"processed/{subset}",
        root / "datasets/dinooooo2006/dataset810",
        root,
    ]
    for d in search_dirs:
        if d.exists():
            for n in names:
                p = d / n
                if p.exists():
                    return p

    for n in names:
        matches = sorted(root.rglob(n)) if root.exists() else []
        if not matches:
            matches = sorted(Path.cwd().rglob(n))
        if subset and matches:
            subset_matches = [p for p in matches if subset.lower() in str(p).lower()]
            if subset_matches:
                return subset_matches[0]
        if matches:
            return matches[0]

    if required:
        raise FileNotFoundError(f"Cannot find any of {names} for subset '{subset}' under {root} or workspace.")
    return None


def resolve_paths(root, scenario, subset="joint"):
    root = Path(root)
    original = find_dataset_file(root, ["train.csv", "train_original_fixed.csv"], subset=subset)
    val = find_dataset_file(root, ["val.csv", "validation_original_fixed.csv"], subset=subset)
    test = find_dataset_file(root, ["test.csv"], subset=subset)
    mlb = find_dataset_file(root, ["multilabel_binarizer.pkl"], subset=subset)
    generic = find_dataset_file(root, [
        "train_augmented_generic_eda.csv",
        "train_augmented_wordnet_eda.csv",
        "train_generic_eda.csv",
        "train_wordnet_eda.csv",
    ], subset=subset, required=False)
    cyber = find_dataset_file(root, [
        "train_augmented_eda.csv",
        "train_augmented_cyber_eda.csv",
        "train_cyber_eda.csv",
        "train_augmented_config4.csv",
    ], subset=subset, required=False)
    canonical_full = find_dataset_file(root, [
        "train_original_full.csv",
        "train_official_original.csv",
        "official_train.csv",
    ], subset=subset, required=False)

    if scenario in {"G0", "G1"} and generic is None:
        raise FileNotFoundError(f"Generic EDA file not found for subset '{subset}'.")
    if scenario in {"B1", "B2_E1"} and cyber is None:
        raise FileNotFoundError(f"Cyber EDA file not found for subset '{subset}'.")

    stage1 = {
        "A0": original,
        "G0": generic,
        "B1": cyber,
        "G1": generic,
        "B2_E1": cyber,
    }[scenario]

    return {
        "original": original,
        "val": val,
        "test": test,
        "mlb": mlb,
        "stage1": stage1,
        "stage2": original if scenario in {"G1", "B2_E1"} else None,
        "canonical_full": canonical_full,
    }


def load_mlb(path):
    with open(path, "rb") as f:
        mlb = pickle.load(f)
    classes = np.asarray(mlb.classes_).astype(str)
    num_labels = len(classes)
    print(f"[MLB] Loaded {num_labels} classes from {path}")
    return mlb, classes, num_labels


def labels_to_matrix(series, mlb):
    return mlb.transform([parse_labels(x) for x in series]).astype(np.uint8)


def run_sanity_checks(train_df, original_df, val_df, test_df):
    tc = detect_col(train_df, ["Text", "text", "Sentence", "sentence", "Cleaned_Text"])
    vc = detect_col(val_df, ["Text", "text", "Sentence", "sentence", "Cleaned_Text"])
    xc = detect_col(test_df, ["Text", "text", "Sentence", "sentence", "Cleaned_Text"])
    s_train = detect_col(train_df, ["source_sample_id"], False)
    s_val = detect_col(val_df, ["source_sample_id"], False)
    s_test = detect_col(test_df, ["source_sample_id"], False)

    if s_train and s_val:
        overlap = set(train_df[s_train].dropna().astype(str)) & set(val_df[s_val].dropna().astype(str))
        if overlap:
            raise AssertionError(f"train/val source ID leakage: {len(overlap)}")
    if s_train and s_test:
        overlap = set(train_df[s_train].dropna().astype(str)) & set(test_df[s_test].dropna().astype(str))
        if overlap:
            raise AssertionError(f"train/test source ID leakage: {len(overlap)}")
    if s_val and s_test:
        overlap = set(val_df[s_val].dropna().astype(str)) & set(test_df[s_test].dropna().astype(str))
        if overlap:
            raise AssertionError(f"val/test source ID leakage: {len(overlap)}")

    tr = set(train_df[tc].map(normalize_text))
    va = set(val_df[vc].map(normalize_text))
    te = set(test_df[xc].map(normalize_text))
    if tr & va:
        raise AssertionError(f"Normalized train/val overlap: {len(tr & va)}")
    if tr & te:
        raise AssertionError(f"Normalized train/test overlap: {len(tr & te)}")
    if va & te:
        raise AssertionError(f"Normalized val/test overlap: {len(va & te)}")

    if len(test_df) == 0:
        raise AssertionError("Test dataset is empty")

    av = detect_col(val_df, ["is_augmented"], False)
    at = detect_col(test_df, ["is_augmented"], False)
    if av and val_df[av].fillna(False).astype(bool).any():
        raise AssertionError("Validation contains synthetic rows")
    if at and test_df[at].fillna(False).astype(bool).any():
        raise AssertionError("Test contains synthetic rows")

    aug = detect_col(train_df, ["is_augmented"], False)
    sid = detect_col(train_df, ["source_sample_id"], False)
    osid = detect_col(original_df, ["source_sample_id"], False)
    ltr = detect_col(train_df, ["Labels", "labels", "Label", "label"])
    lor = detect_col(original_df, ["Labels", "labels", "Label", "label"])
    if aug and sid and osid:
        parent = {str(r[osid]): set(parse_labels(r[lor])) for _, r in original_df.iterrows()}
        syn = train_df[train_df[aug].fillna(False).astype(bool)]
        for _, r in syn.iterrows():
            k = str(r[sid])
            if k not in parent:
                raise AssertionError(f"Synthetic parent missing: {k}")
            if set(parse_labels(r[ltr])) != parent[k]:
                raise AssertionError(f"Synthetic label mismatch for parent: {k}")

    print("[OK] Sanity checks passed")


def support_from_df(df, mlb):
    lc = detect_col(df, ["Labels", "labels", "Label", "label"])
    return labels_to_matrix(df[lc], mlb).sum(axis=0).astype(int)


def canonical_frequency_groups(paths, original_df, val_df, mlb, classes):
    if paths["canonical_full"] is not None:
        src = pd.read_csv(paths["canonical_full"])
        support = support_from_df(src, mlb)
        source_name = str(paths["canonical_full"])
    else:
        src = pd.concat([original_df, val_df], ignore_index=True)
        support = support_from_df(src, mlb)
        source_name = "train + val"

    groups = np.where(support >= 100, "Head", np.where(support >= 20, "Medium", "Tail"))
    counts = pd.Series(groups).value_counts().to_dict()
    print(f"[OK] Canonical frequency groups ({source_name}): {counts}")
    return pd.DataFrame({
        "Technique_ID": classes,
        "Original_Train_Support": support,
        "Frequency_Group": groups,
    })


def tokenize_text(text):
    return TOKEN_RE.findall(str(text).lower())


def build_vocab(original_df, max_vocab=50000):
    tc = detect_col(original_df, ["Text", "text", "Sentence", "sentence", "Cleaned_Text"])
    c = Counter()
    for x in original_df[tc].astype(str):
        c.update(tokenize_text(x))
    vocab = {"<PAD>": PAD_IDX, "<UNK>": UNK_IDX}
    for token, _ in c.most_common(max_vocab - 2):
        vocab[token] = len(vocab)
    print(f"[VOCAB] {len(vocab):,} tokens; built from fixed original train only")
    return vocab


def encode_text(text, vocab, max_length):
    ids = [vocab.get(t, UNK_IDX) for t in tokenize_text(text)[:max_length]]
    out = np.zeros(max_length, dtype=np.int32)
    if ids:
        out[:len(ids)] = ids
    return out


class EncodedTextDataset(Dataset):
    def __init__(self, df, vocab, mlb, max_length):
        tc = detect_col(df, ["Text", "text", "Sentence", "sentence", "Cleaned_Text"])
        lc = detect_col(df, ["Labels", "labels", "Label", "label"])
        self.x = np.stack([encode_text(x, vocab, max_length) for x in df[tc].astype(str)])
        self.y = labels_to_matrix(df[lc], mlb)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return torch.from_numpy(self.x[i].astype(np.int64, copy=False)), torch.from_numpy(self.y[i].astype(np.float32, copy=False))


class TextCNN(nn.Module):
    def __init__(self, vocab_size, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, CONFIG["embedding_dim"], padding_idx=PAD_IDX)
        self.convs = nn.ModuleList([
            nn.Conv1d(CONFIG["embedding_dim"], CONFIG["num_filters"], k)
            for k in CONFIG["kernel_sizes"]
        ])
        self.dropout = nn.Dropout(CONFIG["dropout"])
        self.fc = nn.Linear(CONFIG["num_filters"] * len(CONFIG["kernel_sizes"]), num_classes)

    def forward(self, ids):
        x = self.embedding(ids).transpose(1, 2)
        x = torch.cat([torch.max(F.relu(conv(x)), dim=2).values for conv in self.convs], dim=1)
        return self.fc(self.dropout(x))


class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        p = torch.sigmoid(logits)
        pos = p
        neg = 1 - p
        if self.clip:
            neg = (neg + self.clip).clamp(max=1)
        loss = targets * torch.log(pos.clamp_min(self.eps)) + (1 - targets) * torch.log(neg.clamp_min(self.eps))
        pt = pos * targets + neg * (1 - targets)
        weight = torch.pow((1 - pt).clamp_min(0), self.gamma_pos * targets + self.gamma_neg * (1 - targets))
        return -(loss * weight).sum(1).mean()


def check_asl():
    _loss = AsymmetricLoss(gamma_neg=CONFIG["gamma_neg"], gamma_pos=CONFIG["gamma_pos"], clip=CONFIG["asl_clip"], eps=CONFIG["asl_eps"])
    _z = torch.tensor([[5., -5.], [-5., 5.]], requires_grad=True)
    _y = torch.tensor([[1., 0.], [0., 1.]])
    _good = _loss(_z, _y)
    _bad = _loss(-_z, _y)
    _good.backward()
    
    _z_pos_good = torch.tensor([[5., 0.]])
    _z_pos_bad = torch.tensor([[-5., 0.]])
    _y_pos = torch.tensor([[1., 0.]])
    assert _loss(_z_pos_good, _y_pos) < _loss(_z_pos_bad, _y_pos), "ASL positive direction check failed"
    
    _z_neg_good = torch.tensor([[0., -5.]])
    _z_neg_bad = torch.tensor([[0., 5.]])
    _y_neg = torch.tensor([[0., 0.]])
    assert _loss(_z_neg_good, _y_neg) < _loss(_z_neg_bad, _y_neg), "ASL negative direction check failed"
    
    assert torch.isfinite(_good), "ASL loss is not finite"
    assert _good < _bad, "ASL direction check failed"
    assert _z.grad is not None and torch.isfinite(_z.grad).all(), "ASL gradient is not finite"
    print("[OK] ASL finite, direction and gradient checks passed")

check_asl()


def binary_metrics(y, pred):
    mip, mir, mif, _ = precision_recall_fscore_support(y, pred, average="micro", zero_division=0)
    map_, mar, maf, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    _, _, wf, _ = precision_recall_fscore_support(y, pred, average="weighted", zero_division=0)
    return {
        "micro_precision": float(mip), "micro_recall": float(mir), "micro_f1": float(mif),
        "macro_precision": float(map_), "macro_recall": float(mar), "macro_f1": float(maf),
        "weighted_f1": float(wf), "hamming_loss": float(hamming_loss(y, pred)),
        "true_labels_per_sample": float(y.sum(1).mean()),
        "predicted_labels_per_sample": float(pred.sum(1).mean()),
    }


def ranking_metrics(y_true, scores, ks=(3, 5)):
    order = np.argsort(-scores, axis=1)
    true_count = np.maximum(y_true.sum(1), 1)
    out = {}
    for k in ks:
        kk = min(k, scores.shape[1])
        hits = np.take_along_axis(y_true, order[:, :kk], axis=1).sum(1)
        out[f"precision_at_{k}"] = float(np.mean(hits / kk))
        out[f"recall_at_{k}"] = float(np.mean(hits / true_count))
        out[f"hit_at_{k}"] = float(np.mean(hits > 0))
    ranks = []
    aps = []
    for i in range(len(y_true)):
        rel = y_true[i, order[i]].astype(bool)
        pos = np.flatnonzero(rel)
        ranks.append((pos[0] + 1) if len(pos) else scores.shape[1] + 1)
        if len(pos):
            aps.append(np.mean([(j + 1) / (p + 1) for j, p in enumerate(pos)]))
        else:
            aps.append(0.0)
    out["mrr"] = float(np.mean(1 / np.asarray(ranks)))
    out["map"] = float(np.mean(aps))
    return out


@torch.no_grad()
def predict(model, loader):
    model.eval()
    ys, logits = [], []
    for ids, y in loader:
        z = model(ids.to(DEVICE, non_blocking=True))
        ys.append(y.numpy())
        logits.append(z.cpu().numpy())
    y = np.concatenate(ys).astype(np.uint8)
    logits = np.concatenate(logits)
    probs = 1.0 / (1.0 + np.exp(-np.clip(logits, -50, 50)))
    return y, probs, logits


def train_stage(model, train_loader, val_loader, out_dir, name, epochs, lr, patience):
    criterion = AsymmetricLoss(
        gamma_neg=CONFIG["gamma_neg"],
        gamma_pos=CONFIG["gamma_pos"],
        clip=CONFIG["asl_clip"],
        eps=CONFIG["asl_eps"],
    )
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CONFIG["weight_decay"])
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    ckpt = out_dir / f"{name}_best.pt"
    best = -1.0; best_epoch = 0; bad = 0; history = []

    for epoch in range(1, epochs + 1):
        model.train(); total = 0.0; batches = 0; t0 = time.time()
        for ids, y in train_loader:
            ids = ids.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = criterion(model(ids), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
            opt.step()
            total += loss.item(); batches += 1

        vy, vp, _ = predict(model, val_loader)
        vm = binary_metrics(vy, (vp >= 0.5).astype(np.uint8))
        row = {"stage": name, "epoch": epoch, "train_loss": total/max(batches,1), **{f"val_{k}": v for k,v in vm.items()}, "epoch_seconds": time.time()-t0}
        history.append(row)
        print(f"[{name}] {epoch}/{epochs} loss={row['train_loss']:.6f} Micro-F1={vm['micro_f1']:.4f} Macro-F1={vm['macro_f1']:.4f}")

        if vm["macro_f1"] > best + 1e-12:
            best = vm["macro_f1"]; best_epoch = epoch; bad = 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
            if bad >= patience:
                print(f"[EARLY STOP] {name}; best epoch={best_epoch}")
                break

    pd.DataFrame(history).to_csv(out_dir / f"{name}_epoch_history.csv", index=False)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model, history, best_epoch, best


def threshold_grid():
    return np.round(np.arange(CONFIG["threshold_min"], CONFIG["threshold_max"] + 1e-9, CONFIG["threshold_step"]), 2)


def tune_global_threshold(y, p):
    rows=[]; best_micro=(-1,0.5); best_macro=(-1,0.5)
    for t in threshold_grid():
        pred=(p>=t).astype(np.uint8)
        mi=f1_score(y,pred,average="micro",zero_division=0)
        ma=f1_score(y,pred,average="macro",zero_division=0)
        rows.append({"threshold":float(t),"micro_f1":mi,"macro_f1":ma})
        if mi>best_micro[0]: best_micro=(mi,float(t))
        if ma>best_macro[0]: best_macro=(ma,float(t))
    return best_micro[1], best_macro[1], pd.DataFrame(rows)


def f2_binary(y, pred):
    tp=((y==1)&(pred==1)).sum(); fp=((y==0)&(pred==1)).sum(); fn=((y==1)&(pred==0)).sum()
    return 0.0 if tp==0 else float(5*tp/(5*tp+4*fn+fp))


def tune_minority_f2(y, p, classes, canonical, global_t, num_labels):
    support=y.sum(0)
    gmap=canonical.set_index("Technique_ID")["Frequency_Group"].to_dict()
    thresholds=np.full(num_labels, global_t, dtype=np.float32)
    fallback=np.ones(num_labels, dtype=bool)
    bestf=np.zeros(num_labels)
    grid=threshold_grid()
    for j,label in enumerate(classes):
        if gmap.get(label, "Tail") not in set(CONFIG["minority_groups"]):
            continue
        if support[j] < CONFIG["min_val_support_per_label"]:
            continue
        scores=[f2_binary(y[:,j], (p[:,j]>=t).astype(np.uint8)) for t in grid]
        k=int(np.argmax(scores)); thresholds[j]=grid[k]; fallback[j]=False; bestf[j]=scores[k]
    table=canonical.copy(); table["Validation_Support"]=support; table["Optimal_Threshold"]=thresholds; table["Validation_F2"]=bestf; table["Fallback_Used"]=fallback
    return thresholds, fallback, table


def make_per_label(y, pred, classes, canonical, stage1_support, val_support, thresholds, fallback):
    tp=((y==1)&(pred==1)).sum(0); fp=((y==0)&(pred==1)).sum(0); fn=((y==1)&(pred==0)).sum(0)
    p=np.divide(tp,tp+fp,out=np.zeros_like(tp,dtype=float),where=(tp+fp)!=0)
    r=np.divide(tp,tp+fn,out=np.zeros_like(tp,dtype=float),where=(tp+fn)!=0)
    f=np.divide(2*p*r,p+r,out=np.zeros_like(p),where=(p+r)!=0)
    d=canonical.set_index("Technique_ID").loc[classes].reset_index()
    d["Augmented_Train_Support"]=stage1_support; d["Validation_Support"]=val_support; d["Test_Support"]=y.sum(0)
    d["TP"]=tp; d["FP"]=fp; d["FN"]=fn; d["Precision"]=p; d["Recall"]=r; d["F1"]=f; d["Threshold"]=thresholds; d["Threshold_Fallback"]=fallback
    return d


def group_summary(d):
    rows=[]
    for g in ["Head","Medium","Tail"]:
        x=d[d["Frequency_Group"]==g]
        rows.append({
            "Frequency_Group":g,"n_labels":len(x),
            "mean_precision":x.Precision.mean() if len(x) else 0.0,"median_precision":x.Precision.median() if len(x) else 0.0,
            "mean_recall":x.Recall.mean() if len(x) else 0.0,"median_recall":x.Recall.median() if len(x) else 0.0,
            "mean_f1":x.F1.mean() if len(x) else 0.0,"median_f1":x.F1.median() if len(x) else 0.0,
            "f1_zero_count":int((x.F1==0).sum()),"f1_zero_percentage":float((x.F1==0).mean()*100) if len(x) else 0.0,
            "f1_one_count":int((x.F1==1).sum()),"f1_one_percentage":float((x.F1==1).mean()*100) if len(x) else 0.0,
            "total_test_support":int(x.Test_Support.sum()),
        })
    return pd.DataFrame(rows)


def run_scenario(SCENARIO, DATASET_SUBSET="joint", DATASET_ROOT="/kaggle/input", RESULTS_ROOT="results"):
    assert SCENARIO in {"A0","G0","B1","G1","B2_E1"}
    set_seed(SEED)
    paths=resolve_paths(DATASET_ROOT, SCENARIO, subset=DATASET_SUBSET)
    print(f"Resolved paths (SCENARIO={SCENARIO}, SUBSET={DATASET_SUBSET}):")
    for k,v in paths.items(): print(f"  {k}: {v}")

    original=pd.read_csv(paths["original"])
    val=pd.read_csv(paths["val"])
    test=pd.read_csv(paths["test"])
    stage1=pd.read_csv(paths["stage1"])
    mlb,classes,num_labels=load_mlb(paths["mlb"])
    CONFIG["num_labels"]=num_labels

    run_sanity_checks(stage1, original, val, test)
    canonical=canonical_frequency_groups(paths, original, val, mlb, classes)
    vocab=build_vocab(original, CONFIG["max_vocab"])

    subset_name = DATASET_SUBSET.replace("_", "-") if DATASET_SUBSET == "cti_to_mitre" else DATASET_SUBSET
    run_dir=Path(RESULTS_ROOT)/"TextCNN"/f"{SCENARIO}_{subset_name}"/f"seed_{SEED}"
    run_dir.mkdir(parents=True, exist_ok=True); (run_dir/"figure_data").mkdir(exist_ok=True); (run_dir/"figures").mkdir(exist_ok=True)
    canonical.to_csv(run_dir/"canonical_frequency_groups.csv",index=False)
    with open(run_dir/"vocab.json","w",encoding="utf-8") as f: json.dump(vocab,f,ensure_ascii=False)

    def loader(df, bs, shuffle):
        ds=EncodedTextDataset(df,vocab,mlb,CONFIG["max_length"])
        return DataLoader(ds,batch_size=bs,shuffle=shuffle,num_workers=CONFIG["num_workers"],pin_memory=torch.cuda.is_available())

    tr1=loader(stage1,CONFIG["batch_size"],True); vl=loader(val,CONFIG["eval_batch_size"],False); tl=loader(test,CONFIG["eval_batch_size"],False)
    model=TextCNN(len(vocab), num_classes=num_labels).to(DEVICE)
    total_params=sum(p.numel() for p in model.parameters()); trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"TextCNN params={total_params:,} (num_classes={num_labels})")

    t0=time.time()
    model,h1,e1,m1=train_stage(model,tr1,vl,run_dir,"stage1",CONFIG["stage1_epochs"],CONFIG["stage1_lr"],CONFIG["early_stopping_patience_stage1"])
    h2=[]; e2=None; m2=None
    if SCENARIO in {"G1","B2_E1"}:
        run_sanity_checks(original, original, val, test)
        tr2=loader(original,CONFIG["batch_size"],True)
        model,h2,e2,m2=train_stage(model,tr2,vl,run_dir,"stage2",CONFIG["stage2_epochs"],CONFIG["stage2_lr"],CONFIG["early_stopping_patience_stage2"])
    training_seconds=time.time()-t0

    vy,vp,_=predict(model,vl)
    global_t,macro_t,sweep=tune_global_threshold(vy,vp); sweep.to_csv(run_dir/"threshold_sweep.csv",index=False)
    if SCENARIO in {"G1","B2_E1"}:
        thresholds,fallback,tab=tune_minority_f2(vy,vp,classes,canonical,global_t,num_labels); tab.to_csv(run_dir/"minority_f2_thresholds.csv",index=False); decision="Minority-F2 Dynamic Threshold"
    else:
        thresholds=np.full(num_labels,global_t,dtype=np.float32); fallback=np.ones(num_labels,dtype=bool); decision="Global Threshold"

    inf0=time.time(); ty,tp,tlogits=predict(model,tl); inference_seconds=time.time()-inf0
    pred=(tp>=thresholds.reshape(1,-1)).astype(np.uint8)
    overall=binary_metrics(ty,pred)
    overall.update(ranking_metrics(ty, tp, ks=(3, 5)))

    lc=detect_col(stage1,["Labels","labels","Label","label"])
    stage1_support=labels_to_matrix(stage1[lc],mlb).sum(0); val_support=vy.sum(0)
    per=make_per_label(ty,pred,classes,canonical,stage1_support,val_support,thresholds,fallback)
    per.to_csv(run_dir/"per_label_metrics.csv",index=False)
    groups=group_summary(per); groups.to_csv(run_dir/"frequency_group_performance.csv",index=False)

    tail=per[(per.Frequency_Group=="Tail")&(per.Test_Support>0)]
    tail_summary={
        "n_tail_test_present":int(len(tail)),
        "mean_precision":float(tail.Precision.mean()) if len(tail) else 0.0,
        "mean_recall":float(tail.Recall.mean()) if len(tail) else 0.0,
        "mean_f1":float(tail.F1.mean()) if len(tail) else 0.0,
        "median_f1":float(tail.F1.median()) if len(tail) else 0.0,
        "f1_zero_count":int((tail.F1==0).sum()),
        "f1_zero_percentage":float((tail.F1==0).mean()*100) if len(tail) else 0.0,
    }

    np.savez_compressed(run_dir/"predictions.npz",y_true=ty,probs=tp,logits=tlogits,thresholds=thresholds)
    pd.DataFrame(h1+h2).to_csv(run_dir/"epoch_history.csv",index=False)
    pd.DataFrame([{"scenario":SCENARIO,"dataset_subset":DATASET_SUBSET,"stage1_rows":len(stage1),"stage2_rows":len(original) if SCENARIO in {"G1","B2_E1"} else 0,"validation_rows":len(val),"test_rows":len(test),"vocab_size":len(vocab),"num_labels":num_labels}]).to_csv(run_dir/"dataset_statistics.csv",index=False)
    pd.DataFrame([{"scenario":SCENARIO,"dataset_subset":DATASET_SUBSET,"training_seconds":training_seconds,"inference_seconds":inference_seconds,"inference_ms_per_sample":1000*inference_seconds/len(test),"total_params":total_params,"trainable_params":trainable_params,"device":str(DEVICE)}]).to_csv(run_dir/"computational_cost.csv",index=False)

    metrics={
        "scenario":SCENARIO,"dataset_subset":DATASET_SUBSET,"backbone":"TextCNN","loss":"AsymmetricLoss",
        "gamma_neg":CONFIG["gamma_neg"],"gamma_pos":CONFIG["gamma_pos"],"asl_clip":CONFIG["asl_clip"],"asl_eps":CONFIG["asl_eps"],
        "decision_rule":decision,"seed":SEED,
        "global_threshold_micro_optimal":global_t,"global_threshold_macro_optimal":macro_t,
        "best_stage1_epoch":e1,"best_stage1_val_macro_f1_at_0_5":m1,
        "best_stage2_epoch":e2,"best_stage2_val_macro_f1_at_0_5":m2,
        "training_seconds":training_seconds,"inference_seconds":inference_seconds,"inference_ms_per_sample":1000*inference_seconds/len(test),
        "total_params":total_params,"trainable_params":trainable_params,"num_labels":num_labels,
        "test_global":overall,"tail_test_present":tail_summary,"config":CONFIG,
    }
    with open(run_dir/"metrics.json","w",encoding="utf-8") as f: json.dump(metrics,f,indent=2)

    print("\n"+"="*70)
    print(f"FINAL TextCNN {SCENARIO} [{DATASET_SUBSET}] | {decision}")
    print(f"Micro-F1={overall['micro_f1']:.4f} | Macro-F1={overall['macro_f1']:.4f} | Weighted-F1={overall['weighted_f1']:.4f}")
    print(f"Hit@3={overall.get('hit_at_3',0):.4f} | Hit@5={overall.get('hit_at_5',0):.4f} | MRR={overall.get('mrr',0):.4f} | MAP={overall.get('map',0):.4f}")
    print(f"Tail test-present mean F1={tail_summary['mean_f1']:.4f}")
    print(f"Output: {run_dir}")
    print("="*70)
    return metrics


In [ ]:
metrics = run_scenario(SCENARIO, DATASET_SUBSET, DATASET_ROOT, RESULTS_ROOT)
metrics